In [1]:
import os
from utils.dataloader import ROOT
os.chdir(ROOT)  # write data/artifacts/ at the repo root

from pathlib import Path
import re

import pandas as pd

from utils.dataloader import ROOT, load_results
from utils import detection
from utils.tables import (
    clean_removal_by_model_table,
    end_to_end_conditional_table,
    negative_edit_types_table,
    refactoring_effectiveness_by_design_and_model_table,
    refactoring_outcomes_by_smell_table,
)

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 80)

OUTPUT_DIR = Path("data/artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = detection.add_n15_label(load_results())
print(f"{len(df)} rows · models: {sorted(df['model_alias'].unique())}")

11554 rows · models: ['deepseek-v4-flash', 'gemini35-flash', 'gpt54-nano', 'haiku45']


# Refactoring effectiveness and safety

RQ2 asks how effective and safe the models are after they decide to refactor a workflow. This notebook reuses the validated RQ1/n15 scope, the construct-diff artifacts generated for the first repetition, and the manual validation verdicts in `data/validation`.

Metrics:

- **SRR**: manual smell-removal rate over produced ground-truth-positive patches (`resolved_verdict == "removed"`).
- **SVR**: automatic schema-validity rate over produced ground-truth-positive patches (`n_invalid_added == 0`).
- **CCR**: manual construct-consistency rate over produced ground-truth-positive patches (`resolved_construct_ok == "yes"`).
- **NMR**: manually confirmed unnecessary-edit rate over all 285 ground-truth-negative n15 pairs per design/model (`resolved_verdict == "unnecessary"`).


In [2]:
from utils import designs

TABLE_MODELS = [
    ("GPT-5.4-nano", "gpt54-nano"),
    ("Gemini 3.5 Flash", "gemini35-flash"),
    ("Claude Haiku 4.5", "haiku45"),
    ("DeepSeek V4 Flash", "deepseek-v4-flash"),
]
DESIGNS = list(designs.DESIGNS)
SMELL_GROUPS = {
    "Localized": [
        "MutableRunnerVersion",
        "SuboptimalCacheUsage",
        "MissingTokenPermissions",
        "MissingUploadCondition",
        "MissingJobTimeout",
        "MissingPermExternalActions",
        "UnpinnedActionVersion",
        "MultiCommandStep",
        "UnpinnedDependencies",
        "UnsafeArtifactUploadFork",
    ],
    "Context-dependent": [
        "SingleLangVersionCI",
        "MissingConcurrencyPR",
        "MissingPathFilters",
        "MissingForkGuardPR",
        "SingleOSExecution",
        "MissingForkGuardSchedule",
        "MissingConcurrencyBranch",
        "MissingConcurrencyGlobal",
        "UnsafeDeployOnFork",
    ],
}
AXES_BY_EXPECTED_MODIFICATION = {
    "ADD": ("added",),
    "REMOVE": ("removed",),
    "MODIFY": ("changed",),
    "ADD_OR_MODIFY": ("added", "changed"),
    "ADD_OR_MODIFY_OR_REMOVE": ("added", "changed", "removed"),
}

CONSTRUCT_INDEX = ROOT / "data" / "artifacts" / "constructs" / "construct_per_patch_index.csv"
CONSTRUCT_OCCURRENCES = ROOT / "data" / "artifacts" / "constructs" / "construct_per_patch.csv"
MAPPING = ROOT / "data" / "smell_expected_construct_mapping_refined.csv"
MANUAL_VALIDATION = ROOT / "data" / "validation" / "manual_validation_final.csv"

construct_index = pd.read_csv(CONSTRUCT_INDEX)
construct_occurrences = pd.read_csv(CONSTRUCT_OCCURRENCES)
expected_constructs = pd.read_csv(MAPPING)
manual_validation = pd.read_csv(MANUAL_VALIDATION)

construct_index = construct_index[construct_index["rep"].eq(1)].copy()
construct_index = construct_index[construct_index["model"].isin([alias for _, alias in TABLE_MODELS])]
construct_occurrences = construct_occurrences[construct_occurrences["rep"].eq(1)].copy()
construct_occurrences = construct_occurrences[
    construct_occurrences["model"].isin([alias for _, alias in TABLE_MODELS])
]

print(f"patch rows: {len(construct_index):,}")
manual_validation = manual_validation[manual_validation["rep"].eq(1)].copy()
manual_validation = manual_validation[
    manual_validation["model"].isin([alias for _, alias in TABLE_MODELS])
]

print(f"construct occurrence rows: {len(construct_occurrences):,}")
print(f"manual validation rows: {len(manual_validation):,}")

patch rows: 2,027
construct occurrence rows: 4,192
manual validation rows: 1,952


## Metric computation

In [3]:
def pct_count(num: int, den: int) -> str:
    if den == 0:
        return "--"
    return rf"{(num / den) * 100:.2f}\%\,{{\scriptsize({num}/{den})}}"


def pct(value: float) -> str:
    if pd.isna(value):
        return "--"
    return rf"{value * 100:.2f}\%"


def expected_signature_specs(mapping: pd.DataFrame) -> list[tuple[str, str, re.Pattern]]:
    specs = []
    primary = mapping[mapping["expectedness"].eq("PRIMARY")]
    for row in primary.itertuples(index=False):
        axes = AXES_BY_EXPECTED_MODIFICATION.get(row.expected_modification_type, ())
        for axis in axes:
            specs.append((row.smell_pascal, axis, re.compile(re.escape(row.expected_construct))))
    return specs


def add_patch_metric_flags(index: pd.DataFrame, occurrences: pd.DataFrame) -> pd.DataFrame:
    out = index.copy()
    out["svr_pass"] = out["n_invalid_added"].fillna(0).eq(0)

    ccr_keys = set()
    for smell, axis, pattern in expected_signature_specs(expected_constructs):
        rows = occurrences[
            occurrences["smell"].eq(smell)
            & occurrences["axis"].eq(axis)
            & occurrences["construct"].astype(str).str.match(pattern, na=False)
        ]
        ccr_keys.update(zip(rows.file_key, rows.smell, rows.design, rows.model, rows.rep))

    out["patch_key"] = list(zip(out.file_key, out.smell, out.design, out.model, out.rep))
    out["ccr_pass"] = out["patch_key"].isin(ccr_keys)
    out["any_modification"] = (
        out["n_added"].fillna(0) + out["n_removed"].fillna(0) + out["n_changed"].fillna(0)
    ).gt(0)
    return out.drop(columns=["patch_key"])


patch_metrics = add_patch_metric_flags(construct_index, construct_occurrences)
patch_metrics.head()

,file_key,smell,design,model,rep,gt_positive,parseable,n_added,n_removed,n_changed,...,stem,churn_added,churn_removed,refactor_justification,refactor_confidence,refactor_confidence_justification,diff_path,svr_pass,ccr_pass,any_modification
0,0xpolygonmiden@miden-vm@changelog.yml,MissingConcurrencyPR,D1,deepseek-v4-flash,1,True,True,3.0,0.0,0.0,...,0xpolygonmiden@miden-vm@changelog@D1@rep1@missingconcurrencypr@deepseek-v4-f...,4,0,Added a workflow-level `concurrency` block (lines 8-9 in the patched version...,high,"The edit is a small, localized insertion of a concurrency block at the workf...",yaml_diffs/0xpolygonmiden@miden-vm@changelog@D1@rep1@missingconcurrencypr@de...,True,True,True
1,0xpolygonmiden@miden-vm@changelog.yml,MissingJobTimeout,D1,deepseek-v4-flash,1,NaN,True,1.0,0.0,0.0,...,0xpolygonmiden@miden-vm@changelog@D1@rep1@missingjobtimeout@deepseek-v4-flash,1,0,Added `timeout-minutes: 10` to the `changelog-reusable` job. This is the onl...,high,"The edit is minimal, localized to a single job, and clearly removes the smel...",yaml_diffs/0xpolygonmiden@miden-vm@changelog@D1@rep1@missingjobtimeout@deeps...,True,True,True
2,AlexProgrammerDE@SoulFire@dependency-submission.yml,MissingConcurrencyGlobal,D1,deepseek-v4-flash,1,NaN,True,3.0,0.0,0.0,...,AlexProgrammerDE@SoulFire@dependency-submission@D1@rep1@missingconcurrencygl...,4,0,Added a `concurrency:` block at the workflow level with group `${{ github.wo...,high,The edit is localized to adding a single concurrency block at the workflow l...,yaml_diffs/AlexProgrammerDE@SoulFire@dependency-submission@D1@rep1@missingco...,True,True,True
3,AlexProgrammerDE@SoulFire@dependency-submission.yml,MissingJobTimeout,D1,deepseek-v4-flash,1,True,True,1.0,0.0,0.0,...,AlexProgrammerDE@SoulFire@dependency-submission@D1@rep1@missingjobtimeout@de...,1,0,Added `timeout-minutes: 10` to the `dependency-submission` job at line 9 (th...,high,"The edit is a single-line addition of `timeout-minutes` to the job, which is...",yaml_diffs/AlexProgrammerDE@SoulFire@dependency-submission@D1@rep1@missingjo...,True,True,True
4,AlexProgrammerDE@SoulFire@dependency-submission.yml##synth,UnsafeDeployOnFork,D1,deepseek-v4-flash,1,True,True,0.0,0.0,1.0,...,AlexProgrammerDE@SoulFire@dependency-submission@D1@rep1@unsafedeployonfork@d...,1,0,Added an 'if' condition to the 'publish' job that checks 'github.event_name ...,high,The change is a single-line addition of a well-known fork guard condition. I...,yaml_diffs/AlexProgrammerDE@SoulFire@dependency-submission@D1@rep1@unsafedep...,True,True,True


In [4]:
NEGATIVE_PAIRS_PER_CELL = int(
    df[(df["repetition"].eq(1)) & (df["gt_n15"].eq(False))]
    .groupby(["design", "model_alias"])
    .size()
    .min()
)

rows = []
for design in DESIGNS:
    for display_model, model in TABLE_MODELS:
        patch_group = patch_metrics[
            patch_metrics["design"].eq(design) & patch_metrics["model"].eq(model)
        ]
        manual_group = manual_validation[
            manual_validation["design"].eq(design) & manual_validation["model"].eq(model)
        ]

        patch_pos = patch_group[patch_group["gt_positive"].eq(True)]
        manual_pos = manual_group[manual_group["gt_positive"].eq(True)]
        manual_neg = manual_group[manual_group["gt_positive"].eq(False)]

        if len(patch_pos) != len(manual_pos):
            raise ValueError(f"Positive patch/manual mismatch for {design}/{model}")

        srr_num = int(manual_pos["resolved_verdict"].eq("removed").sum())
        svr_num = int(patch_pos["svr_pass"].sum())
        ccr_num = int(manual_pos["resolved_construct_ok"].eq("yes").sum())
        nmr_num = int(manual_neg["resolved_verdict"].eq("unnecessary").sum())
        uncertain_neg = int(manual_neg["resolved_verdict"].eq("uncertain").sum())
        necessary_neg = int(manual_neg["resolved_verdict"].eq("necessary").sum())
        preserved_neg = int(manual_neg["resolved_verdict"].eq("preserved").sum())

        rows.append(
            {
                "design": design,
                "model": model,
                "display_model": display_model,
                "positive_patches": len(manual_pos),
                "SRR_num": srr_num,
                "SRR": srr_num / len(manual_pos) if len(manual_pos) else pd.NA,
                "SVR_num": svr_num,
                "SVR": svr_num / len(patch_pos) if len(patch_pos) else pd.NA,
                "CCR_num": ccr_num,
                "CCR": ccr_num / len(manual_pos) if len(manual_pos) else pd.NA,
                "negative_pairs": NEGATIVE_PAIRS_PER_CELL,
                "negative_modified": len(manual_neg),
                "negative_unnecessary": nmr_num,
                "negative_necessary": necessary_neg,
                "negative_preserved": preserved_neg,
                "negative_uncertain": uncertain_neg,
                "NMR": nmr_num / NEGATIVE_PAIRS_PER_CELL,
            }
        )

rq2_by_design_model = pd.DataFrame(rows)
rq2_by_design_model


,design,model,display_model,positive_patches,SRR_num,SRR,SVR_num,SVR,CCR_num,CCR,negative_pairs,negative_modified,negative_unnecessary,negative_necessary,negative_preserved,negative_uncertain,NMR
0,D1,gpt54-nano,GPT-5.4-nano,201,193,0.960199,201,1.000000,192,0.955224,285,8,7,0,0,1,0.024561
1,D1,gemini35-flash,Gemini 3.5 Flash,222,210,0.945946,221,0.995495,209,0.941441,285,4,3,0,0,1,0.010526
2,D1,haiku45,Claude Haiku 4.5,242,232,0.958678,241,0.995868,233,0.962810,285,18,14,2,0,2,0.049123
3,D1,deepseek-v4-flash,DeepSeek V4 Flash,237,224,0.945148,237,1.000000,222,0.936709,285,15,14,1,0,0,0.049123
4,D2,gpt54-nano,GPT-5.4-nano,234,228,0.974359,234,1.000000,220,0.940171,285,18,15,0,1,2,0.052632
5,D2,gemini35-flash,Gemini 3.5 Flash,228,215,0.942982,228,1.000000,220,0.964912,285,14,8,0,5,1,0.028070
6,D2,haiku45,Claude Haiku 4.5,239,235,0.983264,238,0.995816,232,0.970711,285,16,15,0,0,1,0.052632
7,D2,deepseek-v4-flash,DeepSeek V4 Flash,238,227,0.953782,236,0.991597,225,0.945378,285,18,17,0,0,1,0.059649


## Table: refactoring effectiveness and safety

In [5]:
effectiveness_path = OUTPUT_DIR / "rq2_refactoring_effectiveness_by_design_and_model.tex"
effectiveness_tex = refactoring_effectiveness_by_design_and_model_table(effectiveness_path)
print(effectiveness_path)
print(effectiveness_tex)

data/artifacts/rq2_refactoring_effectiveness_by_design_and_model.tex
\begin{table}[H]
\centering
\scriptsize
\caption{Refactoring effectiveness and safety by design and model.}
\label{tab:rq2_refactoring_effectiveness_by_design_and_model}
\setlength{\tabcolsep}{3.5pt}
\renewcommand{\arraystretch}{1.08}
\begin{adjustbox}{max width=\linewidth}
\begin{tabular}{llrrrrr}
\toprule
\multirow{2}{*}{\textbf{Design}}
& \multirow{2}{*}{\textbf{Model}}
& \multicolumn{4}{c}{\textbf{Ground-truth-positive pairs}}
& \multicolumn{1}{c}{\textbf{Ground-truth-negative pairs}} \\
\cmidrule(lr){3-6}
\cmidrule(lr){7-7}
&
& \textbf{Pos. Patches}
& \textbf{SRR$\uparrow$}
& \textbf{SVR$\uparrow$}
& \textbf{CCR$\uparrow$}
& \textbf{NMR$\downarrow$} \\
\midrule
\multirow{5}{*}{\textbf{D1}}
& GPT-5.4-nano & 201/285 & 67.72\%\,{\scriptsize(193/285)} & 100.00\%\,{\scriptsize(201/201)} & 100.00\%\,{\scriptsize(191/191)} & 2.46\%\,{\scriptsize(7/285)} \\
& Gemini 3.5 Flash & 222/285 & 73.68\%\,{\scriptsize(210/285)} &

## Table: types of unnecessary edits on negative pairs

In [6]:
negative_edit_types_path = OUTPUT_DIR / "rq2_negative_edit_types.tex"
negative_edit_types_tex = negative_edit_types_table(negative_edit_types_path)

## Table: construct-aware refactoring outcomes by smell

In [7]:
outcomes_by_smell_path = OUTPUT_DIR / "rq2_refactoring_outcomes_by_smell.tex"
outcomes_by_smell_tex = refactoring_outcomes_by_smell_table(outcomes_by_smell_path)

## Table: end-to-end and conditional refactoring outcomes

In [8]:
end_to_end_conditional_path = OUTPUT_DIR / "rq2_end_to_end_conditional.tex"
end_to_end_conditional_tex = end_to_end_conditional_table(end_to_end_conditional_path)

## Table: detection misses and clean removal by model

In [9]:
clean_removal_by_model_path = OUTPUT_DIR / "rq2_clean_removal_by_model.tex"
clean_removal_by_model_tex = clean_removal_by_model_table(clean_removal_by_model_path)

## Validation and coverage information

In [10]:
behavior_summary_path = ROOT / "data" / "artifacts" / "validation" / "behavior_summary_rep1.csv"
validation_scope_path = ROOT / "data" / "artifacts" / "validation" / "rq2_validation_scope_summary.csv"

behavior_summary = pd.read_csv(behavior_summary_path)
validation_scope = pd.read_csv(validation_scope_path)
manual_resolution_summary = manual_validation.groupby(
    ["gt_positive", "resolved_verdict"], as_index=False
).size()

print("Validation scope")
print(validation_scope.to_string(index=False))
print("\nBehavior outcomes")
print(behavior_summary.to_string(index=False))
print("\nManual validation outcomes")
print(manual_resolution_summary.to_string(index=False))


Validation scope
                                metric  value
                rep1_gt_scoped_outputs   4560
                  manual_scope_outputs   2432
         manual_scope_positive_outputs   2280
manual_scope_modified_negative_outputs    152
            manual_scope_existing_tree   2409
            manual_scope_existing_diff   2432
         manual_scope_unparseable_tree     23

Behavior outcomes
population       behavior_outcome_by_gt    n
  negative       correctly_not_modified 2128
  negative      unexpected_modification  152
  positive            expected_modified 1873
  positive missed_required_modification  407

Manual validation outcomes
 gt_positive resolved_verdict  size
       False        necessary     3
       False        preserved     6
       False        uncertain     9
       False      unnecessary    93
        True      not removed    38
        True          removed  1764
        True        uncertain    39


In [11]:
summary = rq2_by_design_model.groupby("design", as_index=False).agg(
    positive_patches=("positive_patches", "sum"),
    smell_removed=("SRR_num", "sum"),
    invalid_positive_patches=("SVR_num", lambda values: int(rq2_by_design_model.loc[values.index, "positive_patches"].sum() - values.sum())),
    construct_inconsistent=("CCR_num", lambda values: int(rq2_by_design_model.loc[values.index, "positive_patches"].sum() - values.sum())),
    negative_modified=("negative_modified", "sum"),
    negative_unnecessary=("negative_unnecessary", "sum"),
    negative_uncertain=("negative_uncertain", "sum"),
)
summary["negative_pairs"] = NEGATIVE_PAIRS_PER_CELL * len(TABLE_MODELS)
summary["macro_SRR"] = rq2_by_design_model.groupby("design")["SRR"].mean().to_numpy()
summary["macro_SVR"] = rq2_by_design_model.groupby("design")["SVR"].mean().to_numpy()
summary["macro_CCR"] = rq2_by_design_model.groupby("design")["CCR"].mean().to_numpy()
summary["macro_NMR"] = rq2_by_design_model.groupby("design")["NMR"].mean().to_numpy()
summary


,design,positive_patches,smell_removed,invalid_positive_patches,construct_inconsistent,negative_modified,negative_unnecessary,negative_uncertain,negative_pairs,macro_SRR,macro_SVR,macro_CCR,macro_NMR
0,D1,902,859,2,46,45,38,4,1140,0.952493,0.997841,0.949046,0.033333
1,D2,939,905,3,42,66,55,5,1140,0.963597,0.996853,0.955293,0.048246


## Outputs

The notebook writes the RQ2 LaTeX tables to `data/artifacts/`. Each is produced by its data-driven generator in `utils/tables/`:

- `rq2_refactoring_effectiveness_by_design_and_model.tex`
- `rq2_negative_edit_types.tex`
- `rq2_refactoring_outcomes_by_smell.tex`
- `rq2_end_to_end_conditional.tex`
- `rq2_clean_removal_by_model.tex`